# LLM-as-Judge — Mechanistic Cross-Validation of NANDA-I Mappings

**Stage 3 of the AVC Transcriptomics × NANDA-I pipeline.**

Reads `outputs/nanda/nanda_mapping_completo_threshold65.csv`
(175 pathway–diagnosis pairs from Stage 2) and cross-validates
each pair via mechanistic reasoning generated by Claude Sonnet 4.6.

### Inputs
- `outputs/nanda/nanda_mapping_completo_threshold65.csv`
  Columns: regulacao, via, via_expandida, rank,
           diagnostico_nanda, similaridade
- `outputs/nanda/threshold_calibration.csv`
  Contains pipeline metadata including colapso_confirmado,
  entropia_ratio_medio, spearman_finetuning, pipeline_versao

### Outputs
- `outputs/nanda/nanda_judge_concordancia.csv`
  All 175 pairs with four judge fields appended:
  justificativa, score_concordancia, nivel_confianca, flag_lexical
- `outputs/nanda/judge_metrics_summary.csv`
  Global concordance metrics and stratification counts
- `outputs/figures/judge_concordance_scatter.png`
  Scatter: cosine similarity (Stage 2) vs judge score (Stage 3)

### Prerequisites
- `ANTHROPIC_API_KEY` must be set as environment variable
- Stage 2 must have completed successfully (C1–C7 passing)
- Run cells in order with Shift+Enter

In [ ]:
# ============================================================
# CÉLULA 1 — Install dependencies
# Run once per session. anthropic>=0.20.0 required for
# claude-sonnet-4-6 and Batch API support.
# ============================================================

%pip install anthropic>=0.20.0 pandas numpy scipy matplotlib \
             seaborn tqdm -q
print('✓ Dependências instaladas')

In [ ]:
# ============================================================
# CÉLULA 2 — Authentication and configuration
#
# ANTHROPIC_API_KEY must be set in the environment before
# starting Jupyter:
#   export ANTHROPIC_API_KEY=sk-ant-...
# Or add it to a .env file (gitignored) and load with dotenv.
# ============================================================

from dotenv import load_dotenv
load_dotenv()

import os
import anthropic

_key = os.environ.get('ANTHROPIC_API_KEY', '').strip()

if not _key:
    raise EnvironmentError(
        'ANTHROPIC_API_KEY not found in environment. '
        'Set it with: export ANTHROPIC_API_KEY=sk-ant-...'
    )

client = anthropic.Anthropic(api_key=_key)

# Verify connectivity with a minimal probe call
try:
    _probe = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=10,
        messages=[{'role': 'user', 'content': 'ping'}],
    )
    print('✓ API key valid — connection established')
    print(f'  Model   : claude-sonnet-4-6')
    print(f'  Key tail: ...{_key[-6:]}')
except anthropic.AuthenticationError:
    raise RuntimeError('Invalid API key. Check ANTHROPIC_API_KEY.')
except Exception as e:
    raise RuntimeError(f'Connection error: {e}')

# ── Runtime constants ─────────────────────────────────────────
MODEL          = 'claude-sonnet-4-6'
TEMPERATURE    = 0        # greedy decoding — maximises reproducibility
SEED           = 42       # fixed seed for API-level reproducibility
MAX_TOKENS     = 500      # sufficient for justificativa + JSON fields
MAX_RETRIES    = 3        # retry attempts per pair
RETRY_BASE_S   = 2.0     # exponential backoff base (seconds)

INPUT_CSV      = 'outputs/nanda/nanda_mapping_completo_threshold65.csv'
CALIB_CSV      = 'outputs/nanda/threshold_calibration.csv'
OUTPUT_CSV     = 'outputs/nanda/nanda_judge_concordancia.csv'
METRICS_CSV    = 'outputs/nanda/judge_metrics_summary.csv'
SCATTER_PNG    = 'outputs/figures/judge_concordance_scatter.png'

print(f'\n✓ Configuração:')
print(f'  MODEL        : {MODEL}')
print(f'  TEMPERATURE  : {TEMPERATURE}')
print(f'  SEED         : {SEED}')
print(f'  MAX_RETRIES  : {MAX_RETRIES}')
print(f'  INPUT        : {INPUT_CSV}')
print(f'  OUTPUT       : {OUTPUT_CSV}')

In [ ]:
# ============================================================
# CÉLULA 3 — Load and validate Stage 2 inputs
#
# Verifies all required columns exist, no pairs below threshold,
# and reads pipeline metadata from threshold_calibration.csv.
# ============================================================

import pandas as pd
import numpy as np
import os

# ── Load pairs ────────────────────────────────────────────────
assert os.path.exists(INPUT_CSV), (
    f'Input not found: {INPUT_CSV}\n'
    'Run clinicalbert_vias_AVC.ipynb (Cells 1–13) first.'
)

df_pairs = pd.read_csv(INPUT_CSV, encoding='utf-8')

REQUIRED_COLS = [
    'regulacao', 'via', 'via_expandida',
    'rank', 'diagnostico_nanda', 'similaridade'
]
missing = [c for c in REQUIRED_COLS if c not in df_pairs.columns]
assert not missing, (
    f'Missing columns in input CSV: {missing}\n'
    'Re-run Cells 9 and 13 of clinicalbert_vias_AVC.ipynb '
    'to regenerate the CSV with via_expandida column.'
)

assert (df_pairs['similaridade'] >= 0.65).all(), (
    'Pairs below threshold 0.65 found — input may be corrupted.'
)

n_up   = (df_pairs['regulacao'] == 'Up').sum()
n_down = (df_pairs['regulacao'] == 'Down').sum()

print(f'✓ {len(df_pairs)} pairs loaded')
print(f'  Up: {n_up}  |  Down: {n_down}')
print(f'  Vias únicas: {df_pairs["via"].nunique()}')
print(f'  Diagnósticos únicos: {df_pairs["diagnostico_nanda"].nunique()}')
print(f'  Score range: '
      f'{df_pairs["similaridade"].min():.4f} – '
      f'{df_pairs["similaridade"].max():.4f}')

# ── Load pipeline metadata ────────────────────────────────────
assert os.path.exists(CALIB_CSV), (
    f'Calibration file not found: {CALIB_CSV}'
)

calib = pd.read_csv(CALIB_CSV)
colapso          = calib['colapso_confirmado'].values[0]
entropia_medio   = calib['entropia_ratio_medio'].values[0]
n_vias_colapso   = int(calib['n_vias_em_colapso'].values[0])
spearman_s2      = calib['spearman_finetuning'].values[0]
pipeline_versao  = calib['pipeline_versao'].values[0]
threshold_final  = calib['threshold_final'].values[0]
z_up             = calib['z_up_at_065'].values[0]

print(f'\n✓ Pipeline metadata loaded ({CALIB_CSV}):')
print(f'  pipeline_versao      : {pipeline_versao}')
print(f'  threshold_final      : {threshold_final} (z = {z_up:.1f})')
print(f'  colapso_confirmado   : {colapso}')
print(f'  entropia_ratio_medio : {entropia_medio:.4f}')
print(f'  n_vias_em_colapso    : {n_vias_colapso}/35')
print(f'  spearman_finetuning  : {spearman_s2}')

if colapso:
    print(f'\n  ⚠️  Entropy collapse confirmed in Stage 2.')
    print(f'     Stage 3 applies stricter mechanistic scrutiny.')

# ── Output directory ──────────────────────────────────────────
os.makedirs('outputs/nanda',   exist_ok=True)
os.makedirs('outputs/figures', exist_ok=True)
print(f'\n✓ Output directories ready')

In [ ]:
# ============================================================
# CÉLULA 4 — Prompt architecture
#
# Two-part prompt design:
#   SYSTEM: establishes role, domain expertise, and the central
#           methodological constraint — justificativa must be
#           produced BEFORE score_concordancia to prevent
#           anchoring on the Stage 2 cosine similarity value.
#   USER:   supplies per-pair context: via label, expanded
#           description (genes + mechanism), regulatory direction,
#           enrichment FDR, Stage 2 cosine score (flagged as
#           potentially inflated by lexical overlap), and full
#           NANDA-I diagnosis description.
# ============================================================

SYSTEM_PROMPT = """You are an expert in post-stroke immunobiology \
and NANDA-I nursing diagnosis taxonomy. You are performing a \
mechanistic cross-validation of pathway\u2013diagnosis mappings \
generated by a semantic embedding pipeline (Stage 2) for an \
ischaemic stroke transcriptomics study (GSE16561, peripheral blood).

The Stage 2 pipeline computed cosine similarity between biological \
pathway embeddings and NANDA-I diagnosis label embeddings. \
However, the pipeline has a documented limitation: entropy collapse \
(H/H_max = {entropia_medio:.4f} across all 35 pathways), indicating \
that high cosine scores may reflect lexical co-occurrence rather \
than genuine mechanistic relevance. Your task is to assess each \
pair independently using biological reasoning.

CRITICAL ORDERING RULE: You must produce the justificativa field \
FIRST, before assigning any numeric score. This ordering prevents \
anchoring on the provided cosine similarity value. Do not write \
the score until you have completed the mechanistic reasoning.

Respond ONLY with valid JSON. No markdown fences, no preamble, \
no text outside the JSON object. Required fields:

{{
  "justificativa": "<2-4 sentences describing the specific \
biological mechanism connecting this pathway to this nursing \
diagnosis in the context of acute ischaemic stroke, OR explaining \
the absence of a direct mechanistic link>",
  "score_concordancia": <float 0.0-1.0, where 1.0 = full \
mechanistic alignment with documented causal pathway, 0.0 = no \
identifiable causal relationship>,
  "nivel_confianca": "<'alta' | 'moderada' | 'baixa'>",
  "flag_lexical": <true if you identify that the Stage 2 cosine \
score is likely inflated by shared vocabulary tokens rather than \
conceptual alignment, false otherwise>
}}"""

SYSTEM_PROMPT = SYSTEM_PROMPT.format(entropia_medio=entropia_medio)


def build_user_prompt(row: pd.Series) -> str:
    """
    Constructs the per-pair user prompt from a DataFrame row.

    Supplies: via label, expanded description, regulatory direction
    with biological interpretation, enrichment FDR (if available),
    Stage 2 cosine score (with explicit lexical-inflation warning),
    and full NANDA-I diagnosis text.
    """
    direction_note = (
        "UP-regulated (innate immune activation / neutrophil "
        "degranulation signature post-stroke)"
        if row['regulacao'] == 'Up'
        else
        "DOWN-regulated (post-stroke adaptive immunodepression / "
        "lymphocyte suppression signature)"
    )

    prompt = f"""PATHWAY TO EVALUATE
Label           : {row['via']}
Expanded desc.  : {row['via_expandida']}
Regulation      : {row['regulacao']} \u2014 {direction_note}
Stage 2 score   : {row['similaridade']:.4f} (cosine similarity; \
NOTE: this score may be inflated by lexical overlap between the \
pathway name and diagnosis text \u2014 assess mechanistic relevance \
independently)

NURSING DIAGNOSIS
{row['diagnostico_nanda']}

Produce your justificativa first, then assign score_concordancia, \
nivel_confianca, and flag_lexical. Return only the JSON object."""

    return prompt


# \u2500\u2500 Sanity check: render prompt for first pair \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
_row0 = df_pairs.iloc[0]
_p0   = build_user_prompt(_row0)
print('\u2713 System prompt built')
print(f'  Length: {len(SYSTEM_PROMPT)} chars')
print()
print('\u2713 User prompt sample (pair 1):')
print('-' * 60)
print(_p0)
print('-' * 60)
print(f'  Length: {len(_p0)} chars')
print(f'  Estimated input tokens: ~{(len(SYSTEM_PROMPT)+len(_p0))//4}')

In [ ]:
# ============================================================
# CÉLULA 5 — API call function with retry, parsing, fallback
#
# call_judge(row) → dict with four judge fields.
# On persistent failure: returns error record (score=null)
# so the output CSV is always complete regardless of API errors.
# ============================================================

import json
import time
import re


def parse_judge_response(text: str) -> dict:
    """
    Extracts and validates the JSON object from the LLM response.

    Strips markdown fences if present, parses JSON, and validates
    that all four required fields exist with correct types.
    Raises ValueError if parsing or validation fails.
    """
    # Strip markdown fences if model ignores the no-fence instruction
    clean = re.sub(r'^```(?:json)?\s*', '', text.strip(), flags=re.M)
    clean = re.sub(r'\s*```$', '', clean, flags=re.M).strip()

    data = json.loads(clean)

    required = {
        'justificativa'     : str,
        'score_concordancia': (int, float),
        'nivel_confianca'   : str,
        'flag_lexical'      : bool,
    }
    for field, expected_type in required.items():
        if field not in data:
            raise ValueError(f"Missing field: '{field}'")
        if not isinstance(data[field], expected_type):
            # Attempt coercion for common type mismatches
            if field == 'score_concordancia':
                data[field] = float(data[field])
            elif field == 'flag_lexical':
                if isinstance(data[field], str):
                    data[field] = data[field].lower() in ('true', '1', 'yes')
                else:
                    raise ValueError(
                        f"'{field}' must be boolean, got "
                        f"{type(data[field]).__name__}"
                    )
            else:
                raise ValueError(
                    f"'{field}' wrong type: expected "
                    f"{expected_type}, got {type(data[field]).__name__}"
                )

    score = data['score_concordancia']
    if not (0.0 <= score <= 1.0):
        raise ValueError(
            f"score_concordancia out of range [0,1]: {score}"
        )

    valid_levels = {'alta', 'moderada', 'baixa'}
    if data['nivel_confianca'] not in valid_levels:
        raise ValueError(
            f"nivel_confianca must be one of {valid_levels}, "
            f"got '{data['nivel_confianca']}'"
        )

    return data


def call_judge(row: pd.Series) -> dict:
    """
    Calls the LLM judge for a single pathway\u2013diagnosis pair.

    Returns a dict with keys: justificativa, score_concordancia,
    nivel_confianca, flag_lexical, judge_attempts, judge_error.

    On persistent failure after MAX_RETRIES, returns an error
    record with score_concordancia=None so downstream processing
    can identify and handle failed pairs.
    """
    user_prompt = build_user_prompt(row)
    last_error  = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.messages.create(
                model       = MODEL,
                max_tokens  = MAX_TOKENS,
                temperature = TEMPERATURE,
                system      = SYSTEM_PROMPT,
                messages    = [
                    {'role': 'user', 'content': user_prompt}
                ],
            )
            raw_text = response.content[0].text
            result   = parse_judge_response(raw_text)
            result['judge_attempts'] = attempt
            result['judge_error']    = None
            return result

        except (json.JSONDecodeError, ValueError) as parse_err:
            last_error = f'ParseError (attempt {attempt}): {parse_err}'
            # Parsing errors: retry immediately (model may have
            # added fences or changed format)
            time.sleep(0.5)

        except anthropic.RateLimitError:
            wait = RETRY_BASE_S ** attempt
            last_error = f'RateLimitError (attempt {attempt})'
            time.sleep(wait)

        except anthropic.APIStatusError as api_err:
            wait = RETRY_BASE_S ** attempt
            last_error = (
                f'APIStatusError {api_err.status_code} '
                f'(attempt {attempt}): {api_err.message}'
            )
            time.sleep(wait)

        except Exception as exc:
            last_error = f'UnexpectedError (attempt {attempt}): {exc}'
            time.sleep(RETRY_BASE_S)

    # All retries exhausted — return error record
    return {
        'justificativa'     : f'ERROR after {MAX_RETRIES} attempts: '
                              f'{last_error}',
        'score_concordancia': None,
        'nivel_confianca'   : 'erro',
        'flag_lexical'      : None,
        'judge_attempts'    : MAX_RETRIES,
        'judge_error'       : last_error,
    }


# \u2500\u2500 Smoke test: call judge on pair 1 only \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
print('Running smoke test on pair 1 (single API call)...')
print('Via     :', df_pairs.iloc[0]['via'])
print('Diagnose:', df_pairs.iloc[0]['diagnostico_nanda'][:60] + '...')
print()

_result = call_judge(df_pairs.iloc[0])

print('\u2713 Smoke test passed')
print(f"  score_concordancia : {_result['score_concordancia']}")
print(f"  nivel_confianca    : {_result['nivel_confianca']}")
print(f"  flag_lexical       : {_result['flag_lexical']}")
print(f"  judge_attempts     : {_result['judge_attempts']}")
print()
print('  justificativa:')
print(' ', _result['justificativa'])

In [ ]:
# ============================================================
# CÉLULA 6 — Processing loop with checkpoint and progress bar
#
# Design principles:
# 1. CHECKPOINT RESUME: if OUTPUT_CSV already exists with
#    partial results, skip already-processed pairs and resume
#    from where processing stopped. Safe to re-run after
#    interruption.
# 2. INCREMENTAL WRITE: each result is appended to the CSV
#    immediately after the API call, not held in memory.
#    Prevents data loss on kernel crash.
# 3. RATE AWARENESS: prints estimated cost and time before
#    starting. Pauses 0.3s between calls to respect API
#    rate limits.
# 4. ERROR ISOLATION: failed pairs receive an error record
#    and processing continues. Error count reported at end.
# ============================================================

import time
from tqdm.notebook import tqdm

INTER_CALL_DELAY = 0.3  # seconds between API calls

# \u2500\u2500 Checkpoint: identify already-processed pairs \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
processed_keys = set()
if os.path.exists(OUTPUT_CSV):
    existing = pd.read_csv(OUTPUT_CSV, encoding='utf-8')
    for _, row in existing.iterrows():
        processed_keys.add((row['via'], row['diagnostico_nanda']))
    print(f'\u2713 Checkpoint found: {len(processed_keys)} pairs '
          f'already processed')
    print(f'  Remaining: {len(df_pairs) - len(processed_keys)} pairs')
else:
    print('\u2713 No checkpoint found \u2014 starting fresh run')
    # Write header row
    header_df = pd.DataFrame(columns=[
        'regulacao', 'via', 'via_expandida', 'rank',
        'diagnostico_nanda', 'similaridade',
        'justificativa', 'score_concordancia',
        'nivel_confianca', 'flag_lexical',
        'judge_attempts', 'judge_error',
    ])
    header_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

# \u2500\u2500 Pre-run cost estimate \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
n_remaining = len(df_pairs) - len(processed_keys)
avg_input_tokens  = (len(SYSTEM_PROMPT) + 500) // 4   # ~system+user
avg_output_tokens = MAX_TOKENS // 2                    # ~250 tokens
cost_input  = n_remaining * avg_input_tokens  * 3 / 1_000_000
cost_output = n_remaining * avg_output_tokens * 15 / 1_000_000
cost_total  = cost_input + cost_output
eta_minutes = n_remaining * (INTER_CALL_DELAY + 1.5) / 60

print(f'\n=== Pre-run estimate ===')
print(f'  Pairs to process : {n_remaining}')
print(f'  Est. input cost  : ${cost_input:.3f}')
print(f'  Est. output cost : ${cost_output:.3f}')
print(f'  Est. total cost  : ${cost_total:.2f} (Sonnet 4.6 standard)')
print(f'  Est. duration    : ~{eta_minutes:.0f} minutes')
print(f'  (50% cheaper with Batch API \u2014 see Cell 6b)')
print()
input('Press Enter to start processing, or Ctrl+C to cancel...')

# \u2500\u2500 Main processing loop \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
n_errors  = 0
n_done    = 0
t_start   = time.time()

pairs_to_process = df_pairs[
    df_pairs.apply(
        lambda r: (r['via'], r['diagnostico_nanda'])
                  not in processed_keys,
        axis=1
    )
]

for idx, row in tqdm(
    pairs_to_process.iterrows(),
    total=len(pairs_to_process),
    desc='Judge calls',
    unit='pair',
):
    result = call_judge(row)

    # Build output record
    out_row = {
        'regulacao'         : row['regulacao'],
        'via'               : row['via'],
        'via_expandida'     : row['via_expandida'],
        'rank'              : row['rank'],
        'diagnostico_nanda' : row['diagnostico_nanda'],
        'similaridade'      : row['similaridade'],
        'justificativa'     : result['justificativa'],
        'score_concordancia': result['score_concordancia'],
        'nivel_confianca'   : result['nivel_confianca'],
        'flag_lexical'      : result['flag_lexical'],
        'judge_attempts'    : result['judge_attempts'],
        'judge_error'       : result['judge_error'],
    }

    # Append to CSV immediately (incremental write)
    pd.DataFrame([out_row]).to_csv(
        OUTPUT_CSV, mode='a', header=False,
        index=False, encoding='utf-8'
    )

    if result['judge_error'] is not None:
        n_errors += 1

    n_done += 1
    time.sleep(INTER_CALL_DELAY)

# \u2500\u2500 Run summary \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
elapsed = time.time() - t_start
print(f'\n\u2713 Processing complete')
print(f'  Pairs processed this run : {n_done}')
print(f'  Errors                   : {n_errors}')
print(f'  Elapsed                  : {elapsed/60:.1f} minutes')
print(f'  Output                   : {OUTPUT_CSV}') 

In [ ]:
# ============================================================
# CÉLULA 7 — Load and validate complete judge output
#
# Reads the full OUTPUT_CSV (may include results from multiple
# resumed runs), validates completeness and field types,
# and reports basic statistics before stratification.
# ============================================================

# \u2500\u2500 Load complete output \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
assert os.path.exists(OUTPUT_CSV), (
    f'{OUTPUT_CSV} not found. Run Cell 6 first.'
)

df_judge = pd.read_csv(OUTPUT_CSV, encoding='utf-8')

# \u2500\u2500 Validate completeness \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
n_total      = len(df_pairs)
n_loaded     = len(df_judge)
n_errors     = (df_judge['nivel_confianca'] == 'erro').sum()
n_valid      = n_loaded - n_errors
n_null_score = df_judge['score_concordancia'].isna().sum()

print(f'=== Output validation ===')
print(f'  Expected pairs : {n_total}')
print(f'  Loaded rows    : {n_loaded}')
print(f'  Valid results  : {n_valid}')
print(f'  Error records  : {n_errors}')
print(f'  Null scores    : {n_null_score}')

if n_loaded < n_total:
    print(f'\n  \u26a0\ufe0f  {n_total - n_loaded} pairs missing.')
    print(f'     Re-run Cell 6 to resume from checkpoint.')
elif n_errors > 0:
    print(f'\n  \u26a0\ufe0f  {n_errors} error record(s). '
          f'Re-run Cell 6 to retry failed pairs.')
else:
    print(f'\n  \u2713 All {n_total} pairs successfully judged')

# \u2500\u2500 Score distribution \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
valid = df_judge[df_judge['score_concordancia'].notna()].copy()
valid['score_concordancia'] = valid['score_concordancia'].astype(float)
valid['flag_lexical'] = valid['flag_lexical'].astype(str).str.lower() \
                              .map({'true': True, 'false': False,
                                    '1': True, '0': False}) \
                              .fillna(False)

print(f'\n=== Score distribution (valid pairs only) ===')
print(f'  min   : {valid["score_concordancia"].min():.4f}')
print(f'  max   : {valid["score_concordancia"].max():.4f}')
print(f'  mean  : {valid["score_concordancia"].mean():.4f}')
print(f'  median: {valid["score_concordancia"].median():.4f}')
print(f'  std   : {valid["score_concordancia"].std():.4f}')

print(f'\n=== nivel_confianca distribution ===')
print(valid['nivel_confianca'].value_counts().to_string())

print(f'\n=== flag_lexical ===')
n_lexical = valid['flag_lexical'].sum()
print(f'  flag_lexical = True  : {n_lexical} ({100*n_lexical/len(valid):.1f}%)')
print(f'  flag_lexical = False : {len(valid)-n_lexical}')

# Make df_judge (cleaned) available for next cells
df_judge = valid.copy()
print(f'\n\u2713 df_judge ready ({len(df_judge)} valid pairs)')

In [ ]:
# ============================================================
# CÉLULA 8 — Convergence stratification and global metrics
#
# Three strata (as defined in the paper):
#   HIGH      : score_concordancia >= 0.70 AND flag_lexical = False
#   MODERATE  : 0.40 <= score_concordancia < 0.70 OR
#               (flag_lexical = False AND borderline score)
#   DIVERGENT : score_concordancia < 0.40 OR flag_lexical = True
#
# Global metric: Spearman correlation between Stage 2 cosine
# similarity and Stage 3 judge score. Low rho (<0.40) confirms
# that the two methods capture different relevance dimensions.
# ============================================================

from scipy.stats import spearmanr

# \u2500\u2500 Stratification \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
def assign_stratum(row):
    s   = row['score_concordancia']
    lex = row['flag_lexical']
    if s >= 0.70 and not lex:
        return 'high_concordance'
    elif s < 0.40 or lex:
        return 'divergent'
    else:
        return 'moderate_concordance'

df_judge['stratum'] = df_judge.apply(assign_stratum, axis=1)

strat_counts = df_judge['stratum'].value_counts()
print('=== Convergence stratification ===')
for stratum, count in strat_counts.items():
    pct = 100 * count / len(df_judge)
    print(f'  {stratum:<25s}: {count:3d} pairs ({pct:.1f}%)')

# \u2500\u2500 Spearman correlation Stage 2 vs Stage 3 \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
rho, p_val = spearmanr(
    df_judge['similaridade'],
    df_judge['score_concordancia']
)
print(f'\n=== Stage 2 vs Stage 3 concordance ===')
print(f'  Spearman rho : {rho:.4f}')
print(f'  p-value      : {p_val:.4e}')

if rho < 0.40:
    interp = ('LOW \u2014 embedding and mechanistic reasoning capture '
              'different relevance dimensions. Entropy collapse '
              'in Stage 2 confirmed as methodological finding.')
elif rho < 0.65:
    interp = ('MODERATE \u2014 partial convergence. Pairs with high '
              'cosine similarity receive mixed mechanistic support.')
else:
    interp = ('HIGH \u2014 convergent validity. High cosine similarity '
              'correlates with mechanistic relevance.')
print(f'  Interpretation: {interp}')

# \u2500\u2500 flag_lexical by via \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
lexical_by_via = (
    df_judge.groupby('via')['flag_lexical']
    .agg(n_lexical='sum', n_total='count')
    .assign(pct_lexical=lambda x: 100*x['n_lexical']/x['n_total'])
    .sort_values('pct_lexical', ascending=False)
)
print(f'\n=== flag_lexical by pathway (top 10) ===')
print(lexical_by_via.head(10).to_string())

# \u2500\u2500 Metrics summary dict \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
metrics = {
    'model'                 : MODEL,
    'temperature'           : TEMPERATURE,
    'seed'                  : SEED,
    'pipeline_versao'       : pipeline_versao,
    'n_pairs_total'         : len(df_judge),
    'n_up'                  : int((df_judge['regulacao']=='Up').sum()),
    'n_down'                : int((df_judge['regulacao']=='Down').sum()),
    'spearman_s2_embedding' : spearman_s2,
    'spearman_s2_vs_s3'     : round(rho, 4),
    'pval_spearman_s2_vs_s3': round(float(p_val), 6),
    'score_mean'            : round(df_judge['score_concordancia'].mean(), 4),
    'score_median'          : round(df_judge['score_concordancia'].median(), 4),
    'score_std'             : round(df_judge['score_concordancia'].std(), 4),
    'n_high_concordance'    : int(strat_counts.get('high_concordance', 0)),
    'n_moderate_concordance': int(strat_counts.get('moderate_concordance', 0)),
    'n_divergent'           : int(strat_counts.get('divergent', 0)),
    'n_flag_lexical_true'   : int(df_judge['flag_lexical'].sum()),
    'pct_flag_lexical_true' : round(100*df_judge['flag_lexical'].mean(), 2),
    'entropia_ratio_s2'     : entropia_medio,
}

pd.DataFrame([metrics]).to_csv(
    METRICS_CSV, index=False, encoding='utf-8'
)
print(f'\n\u2713 Metrics exported \u2192 {METRICS_CSV}')

In [ ]:
# ============================================================
# CÉLULA 9 — Scatter plot: cosine similarity vs judge score
#
# Each point is a pathway–diagnosis pair, coloured by stratum.
# Diagonal reference line shows perfect S2–S3 agreement.
# Annotations show Spearman rho and n per stratum.
# Publication-quality (300 dpi, white background).
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

STRATUM_COLORS = {
    'high_concordance'    : '#1D7A3A',
    'moderate_concordance': '#C47A00',
    'divergent'           : '#9B2121',
}
STRATUM_LABELS = {
    'high_concordance'    : 'High concordance',
    'moderate_concordance': 'Moderate concordance',
    'divergent'           : 'Divergent',
}

fig, ax = plt.subplots(figsize=(7, 6.5))

for stratum, color in STRATUM_COLORS.items():
    sub = df_judge[df_judge['stratum'] == stratum]
    n   = len(sub)
    ax.scatter(
        sub['similaridade'],
        sub['score_concordancia'],
        c=color, alpha=0.72, s=38,
        label=f"{STRATUM_LABELS[stratum]} (n={n})",
        edgecolors='none', linewidths=0,
    )

# Reference line: perfect agreement slope
x_ref = np.linspace(
    df_judge['similaridade'].min() - 0.01,
    df_judge['similaridade'].max() + 0.01, 100
)
ax.plot(x_ref, x_ref, color='#888888', linewidth=0.8,
        linestyle='--', label='Perfect agreement (y = x)',
        zorder=0)

# Spearman annotation
ax.text(
    0.04, 0.96,
    f"Spearman \u03c1 = {rho:.3f}  (p = {p_val:.2e})\n"
    f"n = {len(df_judge)} pairs",
    transform=ax.transAxes,
    fontsize=8.5, verticalalignment='top',
    bbox=dict(boxstyle='round,pad=0.4',
              facecolor='white', edgecolor='#CCCCCC',
              linewidth=0.7),
)

ax.set_xlabel(
    'Stage 2 cosine similarity (embedding pipeline)',
    fontsize=9.5
)
ax.set_ylabel(
    'Stage 3 judge score (mechanistic reasoning)',
    fontsize=9.5
)
ax.set_title(
    'Stage 2 vs Stage 3 Concordance\n'
    'Ischaemic stroke pathway \u2013 NANDA-I diagnosis pairs',
    fontsize=10, pad=10
)
ax.legend(fontsize=8, framealpha=0.9, loc='lower right')
ax.set_xlim(
    df_judge['similaridade'].min() - 0.015,
    df_judge['similaridade'].max() + 0.015
)
ax.set_ylim(-0.05, 1.05)
ax.tick_params(labelsize=8.5)
ax.grid(True, linewidth=0.3, alpha=0.5, color='#CCCCCC')
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(SCATTER_PNG, dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()
print(f'\u2713 Figure saved \u2192 {SCATTER_PNG}')

In [ ]:
# ============================================================
# CÉLULA 10 — Final assertions and output summary
#
# Verifies all outputs are present, complete, and internally
# consistent before the results are used downstream.
#
# NOTE: INPUT CSV contains 5 duplicate pairs (via "Immune
# response", Down, ranks 1–5 appear in both Up and Down sets).
# J1 compares against n_unique_input (170), not len(df_pairs)
# (175), to correctly account for this known upstream issue.
# ============================================================

import pandas as pd, os

OUTPUT_CSV  = 'outputs/nanda/nanda_judge_concordancia.csv'
INPUT_CSV   = 'outputs/nanda/nanda_mapping_completo_threshold65.csv'
METRICS_CSV = 'outputs/nanda/judge_metrics_summary.csv'
SCATTER_PNG = 'outputs/figures/judge_concordance_scatter.png'

df_input  = pd.read_csv(INPUT_CSV,  encoding='utf-8')
df_final  = pd.read_csv(OUTPUT_CSV, encoding='utf-8')

# Pares únicos reais no input (exclui 5 duplicatas upstream)
n_unique_input = df_input.drop_duplicates(
    subset=['via', 'diagnostico_nanda', 'rank']
).shape[0]

n_valid_final = df_final[
    df_final['nivel_confianca'] != 'erro'
]['score_concordancia'].notna().sum()

print('=== Final assertions ===')

# J1 — CSV completo (comparado contra únicos do input)
assert n_valid_final == n_unique_input, (
    f'J1 FAIL: expected {n_unique_input} unique pairs, '
    f'got {n_valid_final}.'
)
print(f'J1 ✓  {OUTPUT_CSV} complete ({n_valid_final} pairs)')

# J2 — Metrics CSV presente
assert os.path.exists(METRICS_CSV), (
    f'J2 FAIL: {METRICS_CSV} missing'
)
print(f'J2 ✓  {METRICS_CSV} present')

# J3 — Scatter figure presente
assert os.path.exists(SCATTER_PNG), (
    f'J3 FAIL: {SCATTER_PNG} missing'
)
print(f'J3 ✓  {SCATTER_PNG} present')

# J4 — Scores dentro do intervalo [0, 1]
scores = df_final['score_concordancia'].dropna().astype(float)
assert (scores >= 0.0).all() and (scores <= 1.0).all(), (
    'J4 FAIL: scores outside [0, 1]'
)
print(f'J4 ✓  All scores in [0, 1]')

# J5 — Coluna stratum presente
assert 'df_judge' in globals(), (
    'J5 FAIL: df_judge not found. Run Cell 7 first.'
)
assert 'stratum' in df_judge.columns, (
    'J5 FAIL: stratum column missing. Run Cell 8 first.'
)
print(f'J5 ✓  stratum column present')

# J6 — Sem duplicatas no output
dupes = df_final.duplicated(
    subset=['via', 'diagnostico_nanda', 'rank']
).sum()
assert dupes == 0, f'J6 FAIL: {dupes} duplicate pairs in output'
print(f'J6 ✓  No duplicate pairs')

# ── Output summary ────────────────────────────────────────────
print(f'\n{"="*60}')
print('  STAGE 3 — COMPLETE')
print(f'{"="*60}')
print(f'  Model              : {MODEL} (T={TEMPERATURE}, seed={SEED})')
print(f'  Pipeline version   : {pipeline_versao}')
print(f'  Pairs judged       : {n_valid_final} (of {n_unique_input} unique)')
print(f'  Input duplicates   : {len(df_input) - n_unique_input} '
      f'(via "Immune response" Down, ranks 1–5)')
print(f'  Spearman S2 vs S3  : {rho:.4f} (p={p_val:.2e})')
print(f'  High concordance   : {strat_counts.get("high_concordance", 0)}')
print(f'  Moderate           : {strat_counts.get("moderate_concordance", 0)}')
print(f'  Divergent          : {strat_counts.get("divergent", 0)}')
print(f'  flag_lexical=True  : {int(df_judge["flag_lexical"].sum())}')
print(f'{"="*60}')
print('\n  Outputs:')
print(f'  → {OUTPUT_CSV}')
print(f'  → {METRICS_CSV}')
print(f'  → {SCATTER_PNG}')
print('\n  Next step: review high-concordance pairs as primary')
print('  hypotheses for the Delphi validation study.')
